In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df[["time", "X", "Y", "ndvi"]]

In [ ]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

In [2]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [2]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=Plumas_Boundaries.geometry(), crs='EPSG:5070', scale=5000)

In [ ]:
print(ds)

In [3]:
import xarray as xr
import numpy as np

def add_ndvi(ds: xr.Dataset) -> xr.Dataset:
    # Bands: B5 = NIR, B4 = Red
    nir = ds["SR_B5"].astype("float32")
    red = ds["SR_B4"].astype("float32")

    # Compute NDVI, guard against divide-by-zero
    denom = nir + red
    ndvi = xr.where(denom != 0, (nir - red) / denom, np.nan).astype("float32")

    # Add attributes for clarity
    ndvi = ndvi.assign_attrs(
        {
            "long_name": "Normalized Difference Vegetation Index",
            "standard_name": "NDVI",
            "description": "NDVI = (SR_B5 - SR_B4) / (SR_B5 + SR_B4)",
            "units": "1",
            "source_bands": "SR_B5 (NIR), SR_B4 (Red)",
        }
    )

    # Mutate dataset directly
    return xr.Dataset({"ndvi": ndvi})

In [ ]:
print(ds)

In [ ]:
# 5. Preview the dataset and the results before doing a full run!
# This allows users to inspect the structure and content of the data to ensure it behaves as expected prior to running a full computation.
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
},
    user_function=compute_ndvi,
    preview_dataset=True
)

In [3]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["time", "X", "Y", "ndvi"]
chunks = {"time": 1, "X": 1024, "Y": 512}

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:5070',
    'scale': 1000,
    'chunks': chunks
},
    user_function=compute_ndvi,
    output_template=df_list,
    user_function_kwargs={
    },
    export_kwargs={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "US_NDVI_Tiles_US_1000m",
        "vrt": True,
        #"chunks": chunks,
        "report": True}, 
    dask_mode="custom",
    dask_kwargs={
        "n_workers": 30,
        "threads_per_worker": 1,
        "memory_limit": "2GB"
    }
)

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\google\cloud\storage\_http.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources

KeyboardInterrupt



In [ ]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df_list = ["time", "X", "Y", "ndvi"]
chunks = {"time": 1, "X": 1024, "Y": 512}

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:5070',
    'scale': 100,
},
    user_function=add_ndvi,
    output_template=df_list,
    chunks=chunks,
    user_function_kwargs={
    },
    export_kwargs={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "US_NDVI_Tiles_US_5000m",
        "vrt": True,
        #"chunks": chunks,
        "report": True}, 
    dask_mode="custom",
    dask_kwargs={
        "n_workers": 30,
        "threads_per_worker": 1,
        "memory_limit": "2GB"
    }
)

Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:60547' processes=30 threads=30, memory=55.88 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] Running user function...


2025-09-23 11:42:32,978 - bokeh.server.protocol_handler - ERROR - error handling message
 message: Message 'PATCH-DOC' content: {'events': [{'kind': 'ModelChanged', 'model': {'id': 'p1088'}, 'attr': 'start', 'new': -1.9450000000000003}, {'kind': 'ModelChanged', 'model': {'id': 'p1088'}, 'attr': 'end', 'new': 30.945}]} 
 error: AssertionError()
Traceback (most recent call last):
  File "r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\bokeh\server\protocol_handler.py", line 94, in handle
    work = await handler(message, connection)
  File "r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\bokeh\server\session.py", line 295, in patch
    return connection.session._handle_patch(message, connection)
  File "r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\bokeh\server\connection.py", line 65, in session
    assert self._session is not None
AssertionError
2025-09-23 11:42:34,333 - bokeh.server.protocol_handler - ERROR - error handling message
 messa